# 5 類純災害分類 v2 — EDA 修正版（Kaggle，Run All）

依 [docs/eda_5class_phase1_analysis.md](../docs/eda_5class_phase1_analysis.md) 第一階段 EDA 的行動清單，對 `train_5class_disaster_kaggle.ipynb`（v1）做以下修正。**其餘設定（模型、epochs、optimizer、sampler）刻意與 v1 相同**，以便直接對照。

| # | 改動 | v1 | v2 |
|---|---|---|---|
| P1 | train 去重 | 無 | dHash 近重複去除（預估 ~3.6k 張，占 train 15.7%） |
| P2 | model selection | val overall acc | **val macro-F1** |
| P3 | 增強配方 | RRC(224, scale 0.08–1.0) → CJ(0.3×3) → Rot(15) 後置 | **Rot(15) 前置** → RRC(scale 0.5–1.0) → CJ(0.15/0.15/0.1) |
| P5 | 輸入解析度 | 224 | **256**（`IMG_SIZE` 設回 224 可隔離此因素） |
| P6 | 評估 | test 全量 | test 全量 + **去洩漏版**（剔除與 train dHash 重複的 ~60 張） |

## 使用前
1. 開 **GPU + Internet**；（可選）Secrets 設 `HF_TOKEN`。
2. **Run All**。比 v1 多一段雜湊掃描（約 3–5 分鐘）。
3. 對照重點：
   - test **macro-F1** 與 **Typhoon/Hurricane recall** 是否高於 v1；
   - 混淆矩陣對照 EDA 的 kNN 矩陣（免訓練下限 acc 0.659）——微調 macro-F1 沒有明顯超過 ~0.75 代表訓練流程有問題。

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
HF_DATASET = "QCRI/MEDIC"

# ── 訓練設定（與 v1 相同，便於對照）───────────────────────────────
CNN_EPOCHS    = 20     # 自訓 CNN 從零訓練
CNN_LR        = 1e-3
FT_EPOCHS     = 8      # 預訓練骨幹微調
FT_LR         = 1e-4
BACKBONE      = "efficientnet_b0"   # 可選 "efficientnet_b0" / "resnet50" / "resnet18"

BATCH_SIZE    = 64
NUM_WORKERS   = 4
SEED          = 42
MAX_SAMPLES_PER_CLASS = None  # train 每類抽樣上限（試跑可設小如 500）；None = 全量
MAX_EVAL_PER_CLASS    = None  # val/test 每類抽樣上限；None = 全量
CACHE_IN_MEMORY       = True  # 影像解碼一次存記憶體；RAM 不足時設 False

# ── v2 新增（依 docs/eda_5class_phase1_analysis.md §9）──────────────
IMG_SIZE       = 256             # P5：輸入解析度 224 → 256（設回 224 可隔離此因素）
RESIZE         = IMG_SIZE + 32   # val/test 的 Resize 尺寸（288）
CACHE_MAX_SIDE = IMG_SIZE + 64   # 記憶體快取縮圖上限（320；對應 RESIZE 不至於放大太多）
DEDUP          = "dhash"         # P1：train 去重。"dhash"=近重複+完全重複 / "md5"=僅完全重複 / None=不去重（=v1）
RRC_SCALE      = (0.5, 1.0)      # P3：RandomResizedCrop scale（v1 為預設 (0.08, 1.0)）
CJ_BRIGHT, CJ_CONTRAST, CJ_SAT = 0.15, 0.15, 0.1   # P3：ColorJitter 由 0.3×3 調降

# ── 5 類純災害（與 v1 一致）─────────────────────────────────────────
CLASSES_EN = [
    "Earthquake Damage", "Flood", "Fire",
    "Typhoon or Storm Damage", "Landslide",
]
CLASSES_ZH = [
    "地震或建築損壞", "淹水", "火災",
    "颱風或強風災損", "土石流或坍方",
]
NUM_CLASSES = len(CLASSES_EN)  # 5

MEDIC_MAP = {
    "earthquake": 0, "flood": 1, "fire": 2,
    "hurricane": 3, "landslide": 4,
    # not_disaster / other_disaster 不列入 → MedicDataset 自動 skip
}
print("NUM_CLASSES:", NUM_CLASSES, "->", CLASSES_EN)
print("BACKBONE:", BACKBONE, "| IMG_SIZE:", IMG_SIZE, "| DEDUP:", DEDUP)

In [ ]:
import os, io, time, copy, hashlib, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from torchvision import transforms
import torchvision.models as tvm
from datasets import load_dataset, Image as HFImage
from PIL import ImageFile, Image
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", message="Palette images with Transparency")
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# （可選）HF 認證
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("已以 HF_TOKEN 認證")
except Exception as e:
    print(f"未使用 HF_TOKEN（{type(e).__name__}），改未認證下載")

raw = load_dataset(HF_DATASET)
print(raw)
SPLIT_TRAIN = "train"
SPLIT_DEV   = "validation" if "validation" in raw else ("dev" if "dev" in raw else None)
SPLIT_TEST  = "test" if "test" in raw else None
assert SPLIT_DEV is not None and SPLIT_TEST is not None, f"找不到 dev/test split：{list(raw.keys())}"
DT_NAMES = raw[SPLIT_TRAIN].features["disaster_types"].names
dropped = [n for n in DT_NAMES if n not in MEDIC_MAP]
print("保留 5 類災害；略過（無災害/其他）:", dropped)

## 去重與洩漏掃描（P1 / P6）

對 train / test 的保留影像計算 MD5（位元組級）與 dHash（感知雜湊，抓縮放 / 重壓縮 / 灰階化的近重複）：
- **P1**：train 內同雜湊只留第一張，其餘剔除——去掉 EDA 找到的 ~15.7% 冗餘，降低 WeightedRandomSampler 對重複圖的有效加權與記憶化。
- **P6**：test 中與 train dHash 重複者**照常參與評估**，但另列「去洩漏版」指標供對照。
- val 洩漏僅 ~1%（27 張），不處理。

MD5 與尺寸只讀位元組；dHash 用 JPEG draft 低解析解碼，全程約 3–5 分鐘。

In [ ]:
def dhash64(img_bytes):
    """8x8 dHash；JPEG 用 draft 低解析解碼加速。回傳 16 字元 hex。"""
    with Image.open(io.BytesIO(img_bytes)) as im:
        im.draft("L", (64, 64))
        g = im.convert("L").resize((9, 8), Image.BILINEAR)
        px = np.asarray(g, dtype=np.int16)
    return np.packbits((px[:, 1:] > px[:, :-1]).flatten()).tobytes().hex()

def scan_hashes(split_name):
    """回傳 [(hf_index, md5, dhash), ...]（僅 5 類保留影像）。"""
    ds_nd = raw[split_name].cast_column("image", HFImage(decode=False))
    names = ds_nd.features["disaster_types"].names
    kept = [i for i, dt in enumerate(ds_nd["disaster_types"]) if names[dt] in MEDIC_MAP]
    out, t0 = [], time.time()
    sub = ds_nd.select(kept)
    for k, item in enumerate(sub):
        b = item["image"]["bytes"]
        if b is None:
            with open(item["image"]["path"], "rb") as f:
                b = f.read()
        try:
            dh = dhash64(b)
        except Exception:
            dh = ""
        out.append((kept[k], hashlib.md5(b).hexdigest(), dh))
        if (k + 1) % 5000 == 0:
            print(f"  {split_name}: {k + 1}/{len(kept)}（{time.time() - t0:.0f}s）")
    print(f"{split_name} 掃描完成：{len(kept)} 張（{time.time() - t0:.0f}s）")
    return out

train_h = scan_hashes(SPLIT_TRAIN)
test_h  = scan_hashes(SPLIT_TEST)

# P1：train 去重（同雜湊留第一張）
train_dup_idx, seen = set(), set()
if DEDUP:
    for hf_i, m5, dh in train_h:
        key = dh if DEDUP == "dhash" else m5
        if key and key in seen:
            train_dup_idx.add(hf_i)
        else:
            seen.add(key)

# P6：test 與 train 的 dHash 近重複（評估時另列去洩漏版）
train_dh_set  = {dh for _, _, dh in train_h if dh}
test_leak_idx = {hf_i for hf_i, _, dh in test_h if dh and dh in train_dh_set}

print(f"\nP1 train 去重（mode={DEDUP}）：剔除 {len(train_dup_idx)} 張")
print(f"P6 test 與 train 近重複（dHash）：{len(test_leak_idx)} 張")

In [ ]:
# P3：rotation 移到 crop 之前（黑角會被後續裁切吃掉）、RRC scale 收斂到 (0.5, 1.0)
#     避免裁到只剩路牌/天空、ColorJitter 調降保住顏色判別線索（火災暖色、颱風中性灰）
train_tf = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=RRC_SCALE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=CJ_BRIGHT, contrast=CJ_CONTRAST, saturation=CJ_SAT),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(RESIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
class MedicDataset(Dataset):
    """把 MEDIC HF split 包成 (image_tensor, label5)；套 MEDIC_MAP（7→5）。

    v2：exclude 內的 HF index 直接跳過（train 去重用）。
    每個 split 各自持有獨立 transform，不共享底層 dataset。
    cache=True 時於 __init__ 將影像解碼並縮到 <=CACHE_MAX_SIDE 存記憶體，
    之後每 epoch 只做 augmentation，免重複 JPEG 解碼。
    """
    def __init__(self, hf_split, transform, max_per_class=None, cache=False, exclude=None):
        self.hf_split  = hf_split
        self.transform = transform
        self.samples   = []                 # list of (hf_index, label5)
        self.counts    = [0] * NUM_CLASSES
        exclude = exclude or set()
        names = hf_split.features["disaster_types"].names
        for i, dt in enumerate(hf_split["disaster_types"]):
            if i in exclude:
                continue
            label = MEDIC_MAP.get(names[dt])
            if label is None:
                continue
            # 注意：max_per_class 取每類「前 N 筆」（未洗牌），純為控訓練時間
            if max_per_class is not None and self.counts[label] >= max_per_class:
                continue
            self.counts[label] += 1
            self.samples.append((i, label))

        # 可選：解碼一次存記憶體（搭配 num_workers=0，避免多進程複製快取）
        self.images = None
        if cache:
            self.images = []
            for hf_i, _ in self.samples:
                im = self.hf_split[hf_i]["image"].convert("RGB")
                im.thumbnail((CACHE_MAX_SIDE, CACHE_MAX_SIDE))
                self.images.append(im.copy())
            mb = sum(im.size[0] * im.size[1] * 3 for im in self.images) / 1e6
            print(f"  已快取 {len(self.images)} 張影像於記憶體（約 {mb:.0f} MB）")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.images is not None:
            img = self.images[idx]
        else:
            hf_i, _ = self.samples[idx]
            img = self.hf_split[hf_i]["image"].convert("RGB")
        return self.transform(img), self.samples[idx][1]


train_ds = MedicDataset(raw[SPLIT_TRAIN], train_tf, max_per_class=MAX_SAMPLES_PER_CLASS,
                        cache=CACHE_IN_MEMORY, exclude=train_dup_idx)
val_ds   = MedicDataset(raw[SPLIT_DEV],   val_tf,  max_per_class=MAX_EVAL_PER_CLASS, cache=CACHE_IN_MEMORY)
test_ds  = MedicDataset(raw[SPLIT_TEST],  val_tf,  max_per_class=MAX_EVAL_PER_CLASS, cache=CACHE_IN_MEMORY)

print("Train 類別分布（去重後）：")
for i, (en, n) in enumerate(zip(CLASSES_EN, train_ds.counts)):
    print(f"  [{i}] {en:25s} {n:6d}")
print(f"Train: {len(train_ds)}  Dev: {len(val_ds)}  Test: {len(test_ds)}")

# sqrt-inverse 類別權重 → WeightedRandomSampler（與 v1 相同）
counts   = np.array(train_ds.counts, dtype=float)
class_w  = 1.0 / np.sqrt(np.maximum(counts, 1.0))
sample_w = [class_w[label] for _, label in train_ds.samples]
sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

# 快取在記憶體時用單進程；否則開多 worker 平行解碼
if CACHE_IN_MEMORY:
    _loader_kw = dict(num_workers=0, pin_memory=True)
else:
    _loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0:
        _loader_kw.update(persistent_workers=True, prefetch_factor=4)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **_loader_kw)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **_loader_kw)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **_loader_kw)

In [ ]:
class DisasterCNN_v1(nn.Module):
    """4-block CNN baseline（與 v1 相同架構；GAP 後接分類頭，輸入尺寸彈性）。"""
    def __init__(self, num_classes: int = 5):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
def build_backbone(name, num_classes):
    """ImageNet 預訓練骨幹，換成 num_classes 類分類頭，全層可微調。"""
    if name == "resnet18":
        net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
        net.fc = nn.Linear(net.fc.in_features, num_classes)
    elif name == "resnet50":
        net = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        net.fc = nn.Linear(net.fc.in_features, num_classes)
    elif name == "efficientnet_b0":
        net = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
        net.classifier[1] = nn.Linear(net.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"未知 backbone: {name}")
    return net

def train_model(model_fn, name, epochs, lr):
    """通用訓練（AMP + cosine + label smoothing）。

    v2（P2）：以 val macro-F1 選 best（v1 用 overall acc，會偏向 Earthquake）。
    回傳 (best_val_macro_f1, best_state)。
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model  = model_fn(NUM_CLASSES).to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None
    best_f1, best_state = 0.0, None
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n=== {name} ===  可訓練參數: {n_params:,}")
    for ep in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            opt.zero_grad()
            if scaler is not None:
                with torch.amp.autocast("cuda"):
                    loss = loss_fn(model(imgs), labels)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss = loss_fn(model(imgs), labels); loss.backward(); opt.step()
        model.eval(); preds, trues = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device, non_blocking=True)
                preds.extend(model(imgs).argmax(1).cpu().tolist())
                trues.extend(labels.tolist())
        acc = float((np.array(preds) == np.array(trues)).mean())
        mf1 = float(f1_score(trues, preds, average="macro"))
        if mf1 > best_f1:
            best_f1, best_state = mf1, copy.deepcopy(model.state_dict())
        sched.step()
        print(f"  Epoch {ep:2d}/{epochs}  Val Acc: {acc:.4f}  MacroF1: {mf1:.4f}  ({time.time()-t0:.1f}s)")
    print(f"  → Best Val MacroF1: {best_f1:.4f}")
    return best_f1, best_state


@torch.no_grad()
def eval_on_test(model_fn, state):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model_fn(NUM_CLASSES).to(device); model.load_state_dict(state); model.eval()
    preds, trues = [], []
    for imgs, labels in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        preds.extend(model(imgs).argmax(1).cpu().tolist())
        trues.extend(labels.tolist())
    return np.array(preds), np.array(trues)

In [ ]:
cnn_f1, cnn_state = train_model(DisasterCNN_v1, "DisasterCNN_v1 (5cls v2, from-scratch)", CNN_EPOCHS, CNN_LR)

In [ ]:
bk_f1, bk_state = train_model(lambda nc: build_backbone(BACKBONE, nc), f"{BACKBONE}-FT (5cls v2)", FT_EPOCHS, FT_LR)

In [ ]:
cnn_pred, cnn_true = eval_on_test(DisasterCNN_v1, cnn_state)
bk_pred,  bk_true  = eval_on_test(lambda nc: build_backbone(BACKBONE, nc), bk_state)

# P6：去洩漏版 mask（test 中與 train dHash 重複者剔除後再算一次指標）
clean_mask = np.array([hf_i not in test_leak_idx for hf_i, _ in test_ds.samples])
print(f"test 全量 {len(clean_mask)} 張；去洩漏後 {int(clean_mask.sum())} 張（剔除 {int((~clean_mask).sum())}）\n")

def row(name, yt, yp):
    return {
        "model":          name,
        "test_acc":       round(float((yp == yt).mean()), 4),
        "test_macroF1":   round(float(f1_score(yt, yp, average="macro")), 4),
        "clean_acc":      round(float((yp[clean_mask] == yt[clean_mask]).mean()), 4),
        "clean_macroF1":  round(float(f1_score(yt[clean_mask], yp[clean_mask], average="macro")), 4),
    }

cmp = pd.DataFrame([
    row("自訓 CNN v2 (from-scratch)", cnn_true, cnn_pred),
    row(f"{BACKBONE}-FT v2",          bk_true,  bk_pred),
])
print(cmp.to_string(index=False))
print("\n（對照下限：ImageNet 特徵 kNN baseline acc = 0.659；微調 macro-F1 應明顯 > ~0.75）")

print("\n=== 自訓 CNN — Test（5 類）===")
print(classification_report(cnn_true, cnn_pred, target_names=CLASSES_EN, digits=4))
print(f"=== {BACKBONE}-FT — Test（5 類）===")
print(classification_report(bk_true, bk_pred, target_names=CLASSES_EN, digits=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (title, yt, yp) in zip(axes, [("Custom CNN v2", cnn_true, cnn_pred),
                                      (f"{BACKBONE}-FT v2", bk_true, bk_pred)]):
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES_EN, yticklabels=CLASSES_EN, ax=ax)
    ax.set_title(f"{title} (Test Acc {float((yp == yt).mean()):.3f})")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig("/kaggle/working/5class_cm_v2.png", dpi=120); plt.show()

## 儲存與下載權重

Run All 後，在右側 **Output** 面板或下方連結下載。
互動 session 結束會清掉 `/kaggle/working`，記得當下就下載或 **Save Version**。

In [ ]:
import json
from IPython.display import FileLink, display

torch.save(bk_state, f"/kaggle/working/{BACKBONE}_5class_v2.pth")   # 預訓練骨幹-FT
torch.save(cnn_state, "/kaggle/working/custom_cnn_5class_v2.pth")    # 自訓 CNN

mapping = {
    "classes":      CLASSES_EN,
    "zh_labels":    CLASSES_ZH,
    "class_to_idx": {c: i for i, c in enumerate(CLASSES_EN)},
    "num_classes":  NUM_CLASSES,
    "dataset":      "QCRI/MEDIC disaster_types（5 類，排除 not/other）",
    "version":      "v2",
    "v2_config": {
        "img_size":            IMG_SIZE,
        "dedup":               DEDUP,
        "train_dedup_removed": len(train_dup_idx),
        "test_leak_excluded":  len(test_leak_idx),
        "rrc_scale":           list(RRC_SCALE),
        "color_jitter":        [CJ_BRIGHT, CJ_CONTRAST, CJ_SAT],
        "model_selection":     "val_macro_f1",
    },
}
with open("/kaggle/working/classes_5class_v2.json", "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

print(f"已存：{BACKBONE}_5class_v2.pth / custom_cnn_5class_v2.pth / classes_5class_v2.json")
for p in [f"/kaggle/working/{BACKBONE}_5class_v2.pth",
          "/kaggle/working/custom_cnn_5class_v2.pth",
          "/kaggle/working/classes_5class_v2.json",
          "/kaggle/working/5class_cm_v2.png"]:
    display(FileLink(p))